<a href="https://colab.research.google.com/github/akshithaarakshi-lab/akshitha-cit-java/blob/main/COLLEGE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U openai langchain-community langchain-huggingface langchain-text-splitters faiss-cpu beautifulsoup4 pypdf
import os
from openai import OpenAI
from google.colab import userdata

# LangChain Imports
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("Step 1: Ingesting data from multiple sources...")

# SOURCE A: Hardcoded Text / Paragraphs (The Detective Data)
raw_paragraphs = [
    """Coorg Institute of Technology (CIT) is a private engineering college located in Ponnampet, Kodagu (Coorg), Karnataka. Established in 1999 by the Kodava Education Society, the institute was founded to provide quality technical education to students from the Kodagu region and across India.
CIT features a sprawling 31-to-34-acre lush green campus nestled in the scenic Western Ghats. The institution is headed by Principal Dr. M. Basavaraj, approved by the AICTE, affiliated with Visvesvaraya Technological University (VTU), Belagavi, and recognized by the Government of Karnataka."""
]
# Convert raw Python strings into LangChain Document objects
text_documents = [Document(page_content=text) for text in raw_paragraphs]

# SOURCE B: Web Scraping (The College Data)
urls = ["https://www.citcoorg.edu.in/cv-admission-form/"]
web_loader = WebBaseLoader(urls)
web_documents = web_loader.load()

# SOURCE C: PDF Documents
pdf_path = "CIT_BROCHURE.pdf"
try:
    pdf_loader = PyPDFLoader(pdf_path)
    pdf_documents = pdf_loader.load()
    print("PDF loaded successfully.")
except Exception as e:
    print(f"Warning: PDF '{pdf_path}' not found in Colab. Skipping PDF ingestion.")
    pdf_documents = []

# MERGE ALL SOURCES
all_documents = text_documents + web_documents + pdf_documents
print(f"Total raw pages/documents loaded: {len(all_documents)}")

# Step 2: Data Chunking
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(all_documents)
print(f"Data chopped into {len(chunks)} searchable chunks.")

# Step 3: Embeddings & FAISS Vector Database
print("Building FAISS Vector Database...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = FAISS.from_documents(chunks, embeddings)

# Step 4: Initialize LLM using the restored GROQ_API_KEY secret name
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=userdata.get("aksh")
)

print("\n817 Unified Universal Bot Online. Ask about the crime scene, college, or PDF.")
print("-" * 60)

# Step 5: The Chat Loop
while True:
    user_query = input("\nYou: ")
    if user_query.lower() == 'exit':
        break

    # Semantic Search across ALL sources simultaneously
    relevant_chunks = vector_db.similarity_search(user_query, k=3)
    retrieved_context = "\n\n".join([doc.page_content for doc in relevant_chunks])

    system_prompt = f"""
    You are an AI assistant powered by a multi-source knowledge base.
    Answer the user's question using ONLY the provided context.
    If the context does not contain the answer, say "I do not have enough information."
    Do NOT guess.

    CONTEXT:
    {retrieved_context}
    """

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_query}
        ],
        temperature=0.0
    )

    print(f"AI: {response.choices[0].message.content}")

Step 1: Ingesting data from multiple sources...
Total raw pages/documents loaded: 2
Data chopped into 13 searchable chunks.
Building FAISS Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


817 Unified Universal Bot Online. Ask about the crime scene, college, or PDF.
------------------------------------------------------------

You: cit
AI: Coorg Institute of Technology (CIT) is a private engineering college located in Ponnampet, Kodagu (Coorg), Karnataka. It was established in 1999 by the Kodava Education Society to provide quality technical education to students from the Kodagu region and across India.  

Key facts:  
- **Campus:** 31‑34 acres of lush green land in the scenic Western Ghats.  
- **Leadership:** Principal Dr. M. Basavaraj.  
- **Accreditation:** Approved by the AICTE, affiliated with Visvesvaraya Technological University (VTU), Belagavi, and recognized by the Government of Karnataka.  

Academic departments include Computer Science & Engineering, Mechanical Engineering, Electronics & Communication, Civil Engineering, Artificial Intelligence and Machine Learning, Artificial Intelligence & Data Science Engineering, Computer Science & Engineering (Cyber Se